# Global Superstore Sales Analysis

## Project Overview
End-to-end data analysis of **51,290+ global retail orders** across multiple markets, regions, and product categories.

**Objectives:**
- Identify top-performing categories, regions, and products
- Analyze profit margins and discount impact
- Detect loss-making segments and customers
- Engineer new features for deeper insights

**Technologies:** Python, Pandas, Matplotlib, SQL Server, Power BI, Excel

## 1. Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load the dataset
df = pd.read_csv("../Data/Global_Superstore2.csv", encoding="latin1")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Display first 5 rows
df.head()

In [ ]:
# Check dataset shape and column names
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Check data types
print(df.dtypes)

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print("Missing percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Basic statistics
df.describe()

## 3. Data Cleaning

**Issues identified:**
-  — 41,296 missing values (80%+ missing) → Drop column
-  &  — String format → Convert to datetime
- No duplicate rows found

In [ ]:
# Drop Postal Code column — 80%+ missing values, not useful for analysis
df = df.drop(columns=["Postal Code"])
print("Postal Code column dropped.")
print("New shape:", df.shape)

In [ ]:
# Convert Order Date and Ship Date from string to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed", dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="mixed", dayfirst=True)

print("Date columns converted successfully!")
print(df[["Order Date", "Ship Date"]].dtypes)

In [ ]:
# Extract Year and Month from Order Date for trend analysis
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month

print("Year and Month extracted successfully!")
print(df[["Order Date", "Order Year", "Order Month"]].head())
print("Shape:", df.shape)

## 4. Feature Engineering

Creating new columns to enhance analysis:
- **Shipping Days** — Number of days taken for delivery
- **Profit Margin** — Profit as a percentage of Sales
- **Is Loss** — Binary flag: 1 if order is a loss, 0 if profit

In [ ]:
# Feature 1: Shipping Days — days between order and delivery
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Feature 2: Profit Margin — profit as % of sales
df["Profit Margin"] = (df["Profit"] / df["Sales"] * 100).round(2)

# Feature 3: Is Loss flag — 1 = loss, 0 = profit
df["Is Loss"] = df["Profit"].apply(lambda x: 1 if x < 0 else 0)

# Verify new features
print("New features created successfully!")
print(df[["Order Date", "Ship Date", "Shipping Days",
          "Sales", "Profit", "Profit Margin", "Is Loss"]].head())

## 5. Save Cleaned & Featured Data

In [ ]:
# Save final cleaned dataset for Power BI and SQL Server
df.to_csv("../Data/Global_Superstore_Final.csv", index=False)

print("Cleaned data saved successfully!")
print("Final Shape:", df.shape)
print("New columns added:", ["Order Year", "Order Month", "Shipping Days", "Profit Margin", "Is Loss"])

## 6. Data Analysis

### 6.1 Category-wise Analysis

In [ ]:
# Category-wise Sales, Profit, Orders, and Profit Margin
category = df.groupby("Category").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count"),
    Avg_Profit_Margin = ("Profit Margin", "mean")
).round(2)

print("Category-wise Analysis:")
print(category.sort_values("Total_Sales", ascending=False))

In [ ]:
# Average discount by Category — to understand discount impact on profit
discount = df.groupby("Category")["Discount"].mean().round(3)
print("Average Discount by Category:")
print(discount.sort_values(ascending=False))

### 6.2 Region-wise Analysis

In [ ]:
# Region-wise Sales, Profit, and Orders
region = df.groupby("Region").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count")
).round(2)

print("Region-wise Analysis:")
print(region.sort_values("Total_Sales", ascending=False))

In [ ]:
# Average discount by Region — to identify high-discount regions
region_discount = df.groupby("Region")["Discount"].mean().round(3)
print("Average Discount by Region:")
print(region_discount.sort_values(ascending=False))

### 6.3 Year-wise Sales Trend

In [ ]:
# Year-wise Sales, Profit, and Orders trend
year_trend = df.groupby("Order Year").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count")
).round(2)

print("Year-wise Sales Trend:")
print(year_trend)

### 6.4 Loss Analysis

In [ ]:
# Identify loss-making orders
loss = df.groupby("Is Loss").agg(
    Total_Orders = ("Order ID", "count"),
    Total_Amount = ("Profit", "sum")
).round(2)

print("Loss vs Profit Analysis:")
print(loss)
print("Loss percentage:", round(df["Is Loss"].sum() / len(df) * 100, 2), "%")

### 6.5 Sub-Category Analysis

In [ ]:
# Top 10 most profitable sub-categories
subcategory = df.groupby("Sub-Category").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count"),
    Avg_Discount = ("Discount", "mean")
).round(2)

print("Top 10 Profitable Sub-Categories:")
print(subcategory.sort_values("Total_Profit", ascending=False).head(10))

In [ ]:
# Bottom 10 loss-making sub-categories
print("Bottom 10 Loss-Making Sub-Categories:")
print(subcategory.sort_values("Total_Profit", ascending=True).head(10))

### 6.6 Top 10 Customers Analysis

In [ ]:
# Top 10 customers by Sales
customers = df.groupby("Customer Name").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count")
).round(2)

print("Top 10 Customers by Sales:")
print(customers.sort_values("Total_Sales", ascending=False).head(10))

### 6.7 Segment-wise Analysis

In [ ]:
# Segment-wise Sales, Profit, and Discount
segment = df.groupby("Segment").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Total_Orders = ("Order ID", "count"),
    Avg_Discount = ("Discount", "mean")
).round(2)

print("Segment-wise Analysis:")
print(segment.sort_values("Total_Sales", ascending=False))

## 7. SQL Server Integration

Importing cleaned data into SQL Server for advanced querying and reporting.

In [ ]:
import pyodbc

# Connect to SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=imraan;"
    "DATABASE=GlobalStoreDB;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
print("Connected to SQL Server successfully!")

In [ ]:
# Load final CSV and import to SQL Server
df_sql = pd.read_csv("../Data/Global_Superstore_Final.csv", encoding="latin1")

# Select required columns for SQL import
df_sql = df_sql[["Order ID", "Order Date", "Ship Date",
                 "Ship Mode", "Customer Name", "Segment",
                 "City", "State", "Country", "Market",
                 "Region", "Category", "Sub-Category",
                 "Product Name", "Sales", "Quantity",
                 "Discount", "Profit", "Shipping Cost",
                 "Order Priority", "Order Year", "Order Month",
                 "Shipping Days", "Profit Margin", "Is Loss"]]

# Insert data row by row into SQL Server
for index, row in df_sql.iterrows():
    cursor.execute("""
        INSERT INTO Orders VALUES
        (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, tuple(row))

conn.commit()
print("Data imported to SQL Server successfully!")
print("Total rows imported:", len(df_sql))

## 7. Data Visualization

Visual analysis of key business metrics across categories, regions, segments, and shipping.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

os.makedirs('../images', exist_ok=True)

# Style settings
BLUE, GREEN, RED, ORANGE, PURPLE = '#2563EB', '#16A34A', '#DC2626', '#EA580C', '#7C3AED'
GRAY, BG, DARK = '#6B7280', '#F8FAFC', '#1E293B'
print('Libraries ready ✅')

In [ ]:
### Chart 1 — Category-wise Sales & Profit
cat = df.groupby('Category').agg(
    Total_Sales=('Sales','sum'),
    Total_Profit=('Profit','sum')
).reset_index().sort_values('Total_Sales', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG)
ax.set_facecolor(BG)
x, w = range(len(cat)), 0.35
bars1 = ax.bar([i - w/2 for i in x], cat['Total_Sales']/1e6, w, color=BLUE, label='Sales ($M)', zorder=3)
bars2 = ax.bar([i + w/2 for i in x], cat['Total_Profit']/1e6, w, color=GREEN, label='Profit ($M)', zorder=3)
for bar in list(bars1)+list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f'${bar.get_height():.2f}M', ha='center', va='bottom', fontsize=11, fontweight='bold', color=DARK)
ax.set_xticks(list(x)); ax.set_xticklabels(cat['Category'], fontsize=13, color=DARK)
ax.set_ylabel('Amount (in Millions $)', fontsize=12); ax.legend(fontsize=11)
ax.set_title('Category-wise Sales & Profit', fontsize=16, fontweight='bold', color=DARK, pad=15)
ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
[spine.set_visible(False) for spine in ax.spines.values()]
plt.tight_layout()
plt.savefig('../images/chart1_category_sales_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 1 saved ✅')

In [ ]:
### Chart 2 — Region-wise Profit Analysis
region = df.groupby('Region').agg(Total_Profit=('Profit','sum')).reset_index()
region = region.sort_values('Total_Profit', ascending=True)
colors = [RED if x < 0 else BLUE for x in region['Total_Profit']]

fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(region['Region'], region['Total_Profit']/1e3, color=colors, zorder=3, height=0.6)
for bar, val in zip(bars, region['Total_Profit']):
    xpos = bar.get_width()
    ax.text(xpos + (1 if xpos >= 0 else -1), bar.get_y()+bar.get_height()/2,
            f'${val/1e3:.1f}K', va='center', ha=('left' if xpos >= 0 else 'right'),
            fontsize=9, fontweight='bold', color=DARK)
ax.axvline(0, color=DARK, linewidth=1.2)
ax.set_xlabel('Total Profit (in Thousands $)', fontsize=12)
ax.set_title('Region-wise Profit Analysis', fontsize=16, fontweight='bold', color=DARK, pad=15)
ax.xaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
[spine.set_visible(False) for spine in ax.spines.values()]
ax.legend(handles=[mpatches.Patch(color=BLUE, label='Profit'), mpatches.Patch(color=RED, label='Loss')], fontsize=11)
plt.tight_layout()
plt.savefig('../images/chart2_region_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 2 saved ✅')

In [ ]:
### Chart 3 — Year-wise Sales & Profit Trend
year = df.groupby('Order Year').agg(Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum')).reset_index()

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG)
ax.set_facecolor(BG)
ax.plot(year['Order Year'], year['Total_Sales']/1e6, marker='o', color=BLUE, linewidth=2.5, markersize=9, label='Sales ($M)')
ax.plot(year['Order Year'], year['Total_Profit']/1e6, marker='s', color=GREEN, linewidth=2.5, markersize=9, label='Profit ($M)')
for _, row in year.iterrows():
    ax.annotate(f"${row['Total_Sales']/1e6:.2f}M", (row['Order Year'], row['Total_Sales']/1e6),
                textcoords='offset points', xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color=BLUE)
    ax.annotate(f"${row['Total_Profit']/1e6:.2f}M", (row['Order Year'], row['Total_Profit']/1e6),
                textcoords='offset points', xytext=(0,-18), ha='center', fontsize=10, fontweight='bold', color=GREEN)
ax.fill_between(year['Order Year'], year['Total_Sales']/1e6, alpha=0.08, color=BLUE)
ax.set_xlabel('Year', fontsize=12); ax.set_ylabel('Amount (in Millions $)', fontsize=12)
ax.set_title('Year-wise Sales & Profit Trend (2011–2014)', fontsize=16, fontweight='bold', color=DARK, pad=15)
ax.set_xticks(year['Order Year']); ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
[spine.set_visible(False) for spine in ax.spines.values()]
plt.tight_layout()
plt.savefig('../images/chart3_yearly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 3 saved ✅')

In [ ]:
### Chart 4 — Top 10 Loss-Making Sub-Categories & Avg Discount
loss_sub = df[df['Is Loss']==1].groupby('Sub-Category').agg(
    Total_Profit=('Profit','sum'), Avg_Discount=('Discount','mean')
).reset_index().sort_values('Total_Profit').head(10)

fig, ax1 = plt.subplots(figsize=(12, 7), facecolor=BG)
ax1.set_facecolor(BG)
bars = ax1.barh(loss_sub['Sub-Category'], loss_sub['Total_Profit']/1e3, color=RED, alpha=0.85, zorder=3, height=0.55)
for bar, val in zip(bars, loss_sub['Total_Profit']):
    ax1.text(bar.get_width()-0.5, bar.get_y()+bar.get_height()/2,
             f'${val/1e3:.1f}K', va='center', ha='right', fontsize=10, fontweight='bold', color='white')
ax2 = ax1.twiny()
ax2.plot(loss_sub['Avg_Discount']*100, loss_sub['Sub-Category'], 'D--', color=ORANGE, linewidth=2, markersize=8)
ax2.set_xlabel('Average Discount (%)', fontsize=11, color=ORANGE)
ax2.tick_params(axis='x', colors=ORANGE)
ax1.set_xlabel('Total Loss (in Thousands $)', fontsize=12)
ax1.set_title('Top 10 Loss-Making Sub-Categories & Avg Discount', fontsize=15, fontweight='bold', color=DARK, pad=15)
ax1.xaxis.grid(True, linestyle='--', alpha=0.4); ax1.set_axisbelow(True)
[spine.set_visible(False) for spine in ax1.spines.values()]
ax1.legend(handles=[mpatches.Patch(color=RED, label='Total Loss'), mpatches.Patch(color=ORANGE, label='Avg Discount %')], fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('../images/chart4_loss_subcategory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 4 saved ✅')

In [ ]:
### Chart 5 — Segment-wise Sales & Profit Distribution
seg = df.groupby('Segment').agg(Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum')).reset_index()
seg_colors = [BLUE, GREEN, PURPLE]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), facecolor=BG)
fig.suptitle('Segment-wise Sales & Profit Distribution', fontsize=16, fontweight='bold', color=DARK)
for ax, col, title in zip(axes, ['Total_Sales','Total_Profit'], ['Sales Distribution','Profit Distribution']):
    ax.set_facecolor(BG)
    wedges, texts, autotexts = ax.pie(seg[col], labels=seg['Segment'], autopct='%1.1f%%',
        colors=seg_colors, startangle=140, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
        textprops=dict(fontsize=12, color=DARK))
    for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold'); at.set_color('white')
    ax.set_title(title, fontsize=13, fontweight='bold', color=DARK, pad=12)
    ax.text(0, 0, f'${seg[col].sum()/1e6:.2f}M\nTotal', ha='center', va='center', fontsize=11, fontweight='bold', color=DARK)
plt.tight_layout()
plt.savefig('../images/chart5_segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 5 saved ✅')

In [ ]:
### Chart 6 — Shipping Mode Order Count & Avg Delivery Days
ship_days = df.copy()
ship_days['Shipping Days'] = (pd.to_datetime(df['Ship Date'], format='mixed', dayfirst=True) -
                               pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True)).dt.days
ship = ship_days.groupby('Ship Mode').agg(Total_Orders=('Order ID','count'), Avg_Days=('Shipping Days','mean')).reset_index()
ship = ship.sort_values('Total_Orders', ascending=False)
ship['Order_Pct'] = ship['Total_Orders'] / ship['Total_Orders'].sum() * 100

fig, ax1 = plt.subplots(figsize=(10, 6), facecolor=BG)
ax1.set_facecolor(BG)
bars = ax1.bar(ship['Ship Mode'], ship['Total_Orders'], color=[BLUE,GREEN,ORANGE,PURPLE], zorder=3, width=0.5)
for bar, pct in zip(bars, ship['Order_Pct']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
             f'{pct:.1f}%', ha='center', fontsize=12, fontweight='bold', color=DARK)
ax2 = ax1.twinx()
ax2.plot(ship['Ship Mode'], ship['Avg_Days'], 'D--', color=RED, linewidth=2.5, markersize=10)
for mode, days in zip(ship['Ship Mode'], ship['Avg_Days']):
    ax2.annotate(f'{days:.1f}d', (mode, days), textcoords='offset points', xytext=(0,10),
                 ha='center', fontsize=10, fontweight='bold', color=RED)
ax2.set_ylabel('Avg Shipping Days', fontsize=12, color=RED)
ax2.tick_params(axis='y', colors=RED)
ax1.set_ylabel('Total Orders', fontsize=12)
ax1.set_title('Shipping Mode — Order Count & Avg Delivery Days', fontsize=15, fontweight='bold', color=DARK, pad=15)
ax1.yaxis.grid(True, linestyle='--', alpha=0.4); ax1.set_axisbelow(True)
[spine.set_visible(False) for spine in ax1.spines.values()]
ax1.legend(handles=[mpatches.Patch(color=BLUE, label='Order Count'), mpatches.Patch(color=RED, label='Avg Shipping Days')], fontsize=10)
plt.tight_layout()
plt.savefig('../images/chart6_shipping_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 6 saved ✅')

## 8. Key Business Insights

### Category Insights
- **Technology** is the top revenue-generating category (.74M) with highest profit (63K)
- **Furniture** has high sales but lowest profit margin — due to high discounts (16.8%)
- **Office Supplies** has the best profit margin (5.9%) despite lower sales

### Region Insights
- **Central** region leads in both sales (.82M) and profit (11K)
- **Southeast Asia** has 84K sales but only 7K profit — 27.2% discount is too high!
- **Turkey** and **Nigeria** are loss-making markets — discount policy needs review

### Year-wise Trend
- Consistent growth from 2011 to 2014
- Sales grew from .26M (2011) to .30M (2014) — 90% growth in 3 years!

### Loss Analysis
- **24.46%** of orders are loss-making — 1 in 4 orders results in loss!
- **Tables** is the only sub-category in overall loss (-4K) due to 29% discount
- **Binders** has the highest discount (59.8%) among loss-making products

### Customer Insights
- **Tom Ashbrook** is the top customer by sales (0K)
- **Sean Miller** has 5K sales but negative profit (-09) — not worth retaining!

### Segment Insights
- **Consumer** segment dominates with 51.48% of total sales (.5M)
- All segments have same discount rate (14%) — pricing strategy is uniform